# CFNN Phase 3 — Visualization & Theory

Publication-quality visualizations across **three datasets** of increasing complexity:

| Dataset | Dims | Task | Why |
|---------|------|------|-----|
| **Moons** | 2 | Binary classification | Interpretable 2D baseline |
| **Wine** | 13 | 3-class classification | Correlated chemical features — CFNN's natural domain |
| **California Housing** | 8 | Regression | Real-world, 20K samples, tests scalability |

For each dataset we train both **CFNN-A** (absorption) and **CFNN-D** (distillation),
then compare how the "column" behaves differently.

**Author:** Daniel Regalado Cardoso | University of Miami

## 1. Setup

In [ ]:
!git clone -b main https://github.com/DanielRegaladoUMiami/counterflow-nn.git
%cd counterflow-nn
!pip install scikit-learn matplotlib pandas pytest torchvision -q

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler

from src.network import CounterFlowNetwork
from src.distillation import DistillationNetwork
from src.utils import load_synthetic_dataset, load_uci_dataset, prepare_data, train_model
from src.visualization import (
    mccabe_thiele_plot, concentration_profile, driving_force_profile,
    transfer_heatmap, diagnostic_dashboard,
    column_schematic, column_schematic_pid, column_schematic_sankey,
)
from src.diagnostics import print_diagnostics

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch {torch.__version__} | Device: {device}')

## 2. Helper: Train CFNN-A and CFNN-D on any dataset

In [ ]:
def train_both_models(
    X, y, dataset_name, task='classification',
    d_gas=32, d_liquid=32, n_plates=8, n_epochs=150,
):
    print(f'\n{"="*60}')
    print(f'  Dataset: {dataset_name}')
    print(f'  Shape: {X.shape} | Task: {task}')
    print(f'{"="*60}')
    train_ld, test_ld, d_in, n_cls = prepare_data(X, y, seed=42)
    X_scaled = torch.FloatTensor(StandardScaler().fit_transform(X))
    print(f'\n--- CFNN-A (Absorption, {n_plates} plates) ---')
    model_a = CounterFlowNetwork(d_in=d_in, d_gas=d_gas, d_liquid=d_liquid,
                                 n_plates=n_plates, d_out=n_cls, n_sweeps=2)
    hist_a = train_model(model_a, train_ld, test_ld, n_epochs=n_epochs,
                         device=device, task=task, verbose=True, print_every=50)
    model_a.to('cpu')
    n_rect = n_plates // 2
    n_strip = n_plates - n_rect
    print(f'\n--- CFNN-D (Distillation, {n_rect}+{n_strip} plates) ---')
    model_d = DistillationNetwork(d_in=d_in, d_gas=d_gas, d_liquid=d_liquid,
                                  n_plates_rect=n_rect, n_plates_strip=n_strip,
                                  d_out=n_cls, n_sweeps=2, reflux_ratio=0.3, reboil_ratio=0.2)
    hist_d = train_model(model_d, train_ld, test_ld, n_epochs=n_epochs,
                         device=device, task=task, verbose=True, print_every=50)
    model_d.to('cpu')
    metric = 'accuracy' if task == 'classification' else 'rmse'
    print(f'\nCFNN-A params: {model_a.count_parameters():,} | {metric}: {hist_a["test_metrics"][-1]:.4f}')
    print(f'CFNN-D params: {model_d.count_parameters():,} | {metric}: {hist_d["test_metrics"][-1]:.4f}')
    return {'name': dataset_name, 'task': task, 'model_a': model_a, 'model_d': model_d,
            'hist_a': hist_a, 'hist_d': hist_d, 'X_scaled': X_scaled, 'd_in': d_in, 'n_cls': n_cls}

---
## 3. Dataset 1 — Moons (2D baseline)

In [ ]:
X_moons, y_moons = load_synthetic_dataset('moons', n_samples=1000, noise=0.2, seed=42)
res_moons = train_both_models(X_moons, y_moons, 'Moons (2D)', n_plates=8)

In [ ]:
fig = mccabe_thiele_plot(res_moons['model_a'], res_moons['X_scaled'],
    title='CFNN-A McCabe-Thiele — Moons', show_steps=True, figsize=(8, 8))
plt.show()
fig = mccabe_thiele_plot(res_moons['model_d'], res_moons['X_scaled'],
    title='CFNN-D McCabe-Thiele — Moons', show_steps=True, figsize=(8, 8))
plt.show()

In [ ]:
fig = concentration_profile(res_moons['model_a'], res_moons['X_scaled'], title='CFNN-A Concentration — Moons')
plt.show()
fig = concentration_profile(res_moons['model_d'], res_moons['X_scaled'], title='CFNN-D Concentration — Moons')
plt.show()

In [ ]:
fig = diagnostic_dashboard(res_moons['model_a'], res_moons['X_scaled'], model_name='CFNN-A — Moons')
plt.show()
fig = diagnostic_dashboard(res_moons['model_d'], res_moons['X_scaled'], model_name='CFNN-D — Moons')
plt.show()

In [ ]:
print('=== CFNN-A Diagnostics (Moons) ===')
print_diagnostics(res_moons['model_a'], res_moons['X_scaled'])
print('\n=== CFNN-D Diagnostics (Moons) ===')
print_diagnostics(res_moons['model_d'], res_moons['X_scaled'])

---
## 4. Dataset 2 — Wine (13D, correlated chemical features)

CFNN's **natural domain**: 13 physicochemical measurements that interact and correlate.

In [ ]:
X_wine, y_wine, _ = load_uci_dataset('wine')
res_wine = train_both_models(X_wine, y_wine, 'Wine (13D)', n_plates=8)

In [ ]:
fig = mccabe_thiele_plot(res_wine['model_a'], res_wine['X_scaled'],
    title='CFNN-A McCabe-Thiele — Wine', show_steps=True, figsize=(8, 8))
plt.show()
fig = mccabe_thiele_plot(res_wine['model_d'], res_wine['X_scaled'],
    title='CFNN-D McCabe-Thiele — Wine', show_steps=True, figsize=(8, 8))
plt.show()

In [ ]:
fig = transfer_heatmap(res_wine['model_a'], res_wine['X_scaled'],
    title='CFNN-A Transfer Heatmap — Wine')
plt.show()
fig = transfer_heatmap(res_wine['model_d'], res_wine['X_scaled'],
    title='CFNN-D Transfer Heatmap — Wine')
plt.show()

In [ ]:
fig = diagnostic_dashboard(res_wine['model_a'], res_wine['X_scaled'], model_name='CFNN-A — Wine')
plt.show()
fig = diagnostic_dashboard(res_wine['model_d'], res_wine['X_scaled'], model_name='CFNN-D — Wine')
plt.show()

---
## 5. Dataset 3 — California Housing (8D, regression)

In [ ]:
X_cal, y_cal, _ = load_uci_dataset('california_housing')
res_cal = train_both_models(X_cal, y_cal, 'California Housing (8D)',
                            task='regression', n_plates=8, n_epochs=100)

In [ ]:
fig = mccabe_thiele_plot(res_cal['model_a'], res_cal['X_scaled'],
    title='CFNN-A McCabe-Thiele — Calif. Housing', show_steps=True, figsize=(8, 8))
plt.show()
fig = mccabe_thiele_plot(res_cal['model_d'], res_cal['X_scaled'],
    title='CFNN-D McCabe-Thiele — Calif. Housing', show_steps=True, figsize=(8, 8))
plt.show()

In [ ]:
fig = driving_force_profile(res_cal['model_a'], res_cal['X_scaled'],
    title='CFNN-A Driving Force — Calif. Housing')
plt.show()
fig = driving_force_profile(res_cal['model_d'], res_cal['X_scaled'],
    title='CFNN-D Driving Force — Calif. Housing')
plt.show()

In [ ]:
fig = transfer_heatmap(res_cal['model_a'], res_cal['X_scaled'],
    title='CFNN-A Transfer Heatmap — Calif. Housing')
plt.show()
fig = transfer_heatmap(res_cal['model_d'], res_cal['X_scaled'],
    title='CFNN-D Transfer Heatmap — Calif. Housing')
plt.show()

In [ ]:
fig = diagnostic_dashboard(res_cal['model_a'], res_cal['X_scaled'], model_name='CFNN-A — Calif. Housing')
plt.show()
fig = diagnostic_dashboard(res_cal['model_d'], res_cal['X_scaled'], model_name='CFNN-D — Calif. Housing')
plt.show()

---
## 6. Column Schematics — P&ID and Flow Diagrams

- **P&ID**: Engineering process diagram with vessel, trays, condenser/reboiler
- **Flow**: Stream widths proportional to norms, cross-flow arrows

In [ ]:
# P&ID Schematics
for res, label in [(res_moons, 'Moons'), (res_wine, 'Wine'), (res_cal, 'Calif. Housing')]:
    fig = column_schematic_pid(res['model_a'], res['X_scaled'], title=f'CFNN-A P&ID — {label}')
    plt.show()
    fig = column_schematic_pid(res['model_d'], res['X_scaled'], title=f'CFNN-D P&ID — {label}')
    plt.show()

In [ ]:
# Flow Diagrams
for res, label in [(res_moons, 'Moons'), (res_wine, 'Wine'), (res_cal, 'Calif. Housing')]:
    fig = column_schematic_sankey(res['model_a'], res['X_scaled'], title=f'CFNN-A Flow — {label}')
    plt.show()
    fig = column_schematic_sankey(res['model_d'], res['X_scaled'], title=f'CFNN-D Flow — {label}')
    plt.show()

---
## 7. Cross-Dataset Comparison

In [ ]:
from src.diagnostics import operating_line_data
fig, axes = plt.subplots(1, 3, figsize=(21, 7))
datasets = [(res_moons, 'Moons (2D)'), (res_wine, 'Wine (13D)'), (res_cal, 'Calif. Housing (8D)')]
for i, (res, label) in enumerate(datasets):
    ax = axes[i]
    op = operating_line_data(res['model_a'], res['X_scaled'])
    gn = op.get('gas_norms', op.get('gas_rect_norms', []))
    ln = op.get('liquid_norms', op.get('liquid_rect_norms', []))
    ml = min(len(gn), len(ln))
    ax.plot(ln[:ml], gn[:ml], 'o-', color='#E74C3C', linewidth=2.5, markersize=10)
    for j in range(ml):
        ax.annotate(f'P{j}', (ln[j], gn[j]), textcoords='offset points', xytext=(8,8), fontsize=9)
    mv = max(max(gn[:ml]), max(ln[:ml])) * 1.1
    ax.plot([0, mv], [0, mv], 'k--', alpha=0.3)
    ax.set_xlabel('||liquid||'); ax.set_ylabel('||gas||')
    ax.set_title(f'CFNN-A: {label}', fontsize=14, fontweight='bold')
    ax.grid(True, alpha=0.2); ax.set_aspect('equal')
plt.suptitle('McCabe-Thiele — Column Adapts to Data Complexity', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
from src.diagnostics import damkohler_number, number_of_transfer_units, alpha_statistics
rows = []
for res, label in datasets:
    for arch, mk in [('model_a','CFNN-A'),('model_d','CFNN-D')]:
        model = res[arch]
        mname = 'accuracy' if res['task']=='classification' else 'rmse'
        mval = res[f'hist_{arch[-1]}']['test_metrics'][-1]
        da = damkohler_number(model, res['X_scaled'])
        ntu = number_of_transfer_units(model, res['X_scaled'])
        alphas = alpha_statistics(model)
        rows.append({'Dataset': label, 'Arch': mk, 'Params': model.count_parameters(),
                     f'{mname}': f'{mval:.4f}', 'Mean Da': f'{np.mean(da["per_plate"]):.4f}',
                     'NTU': f'{ntu["ntu"]:.4f}', 'Mean alpha': f'{alphas["mean_alpha"]:.4f}'})
print(pd.DataFrame(rows).to_string(index=False))

---
## 8. Mathematical Foundations

### 8.1 From Mass Transfer to Neural Networks

$$G_s \frac{dY}{dZ} = K_Y a (Y - Y^*) \quad \Rightarrow \quad \Delta_n = \alpha \cdot \tanh(T(g_n - E(l_n)))$$

### 8.2 Conservation: $\Delta g = -\Delta l$ (hard constraint)

### 8.3 CFNN-D adds: bidirectional transfer ($\alpha$, $\beta$), feed plate (q-line), reflux/reboil

| Metric | ChemE | CFNN |
|--------|-------|------|
| **Da** | Reaction/flow rate | Transfer/info flow |
| **$\eta$** | Actual/equil change | Plate efficiency |
| **NTU** | $\int dY/(Y-Y^*)$ | Effective depth |
| **R** | L/D condenser | Reflux ratio |
| **q** | Liquid fraction feed | Learned split |

---
## Phase 3 Complete!

**Next:** Phase 4 — Gradio Space + HuggingFace Hub deployment.